<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 05. Lottery Ticket Hypothesis

## The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks

- **원 논문:** [The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks](https://openreview.net/forum?id=rJl-b3RcF7)
- **저자 / 발표:** Jonathan Frankle, Michael Carbin · ICLR 2019 (2019)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** dense network 학습으로 mask를 찾고 surviving weight를 초기값으로 되감아 sparse subnetwork를 재학습한다.

**축소하거나 생략한 부분:** 원 논문은 MNIST/CIFAR의 반복 magnitude pruning과 여러 seed를 비교했다. 실습은 한 번의 pruning과 작은 MLP다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2: winning ticket definition | Task 1 | 초기값·mask 보존 |
| §2, Figure 1: iterative pruning procedure | Task 1–2 | train→prune→rewind |
| §3: experimental methodology | Task 2 | 동일 mask·step과 세 fresh optimizer |
| §4: winning tickets results | Task 3 | rewind ticket/random sparse/dense 비교 |

## 미니 재현
중요한 것은 작은 weight 자체뿐 아니라, 그 연결에 대응하는 **초기값으로 되감는 것**이라는 가설을 코드로 분리합니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$f(\mathbf x,m\odot\theta_0)\quad\text{with}\quad\operatorname{Train}(m\odot\theta_0)
\approx\operatorname{Train}(\theta)$$

- **기호 정의:** m은 pruning mask, θ0는 원래 초기값, m⊙θ0는 winning ticket 후보입니다.
- **수식의 역할:** 학습 후 찾은 sparse 연결을 원래 초기값으로 되감아 독립적으로 재학습합니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  §2: winning ticket definition의 흐름을 명시합니다.
- **입출력 shape:** 특징 [B, 32] → masked hidden [B, 64] → class logit [B, 4]
- **평가:** 동일 예산의 rewind ticket/random sparse/dense 정확도와 loss
- **원문 대비 한계:** 원 논문은 MNIST/CIFAR의 반복 magnitude pruning과 여러 seed를 비교했다. 실습은 한 번의 pruning과 작은 MLP다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,np.load,torch.from_numpy -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>np.load(file, allow_pickle=False, ...)</code></summary>

#### `np.load(file, allow_pickle=False, ...)`

- **역할:** NumPy 배열이나 archive를 파일에서 읽습니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.load](https://numpy.org/doc/stable/reference/generated/numpy.load.html)

</details>

<details>
<summary><code>torch.from_numpy(ndarray)</code></summary>

#### `torch.from_numpy(ndarray)`

- **역할:** NumPy 배열과 메모리를 공유하는 CPU 텐서를 만듭니다.
- **입력·반환:** NumPy `ndarray`를 받고 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 복사 없이 준비된 NumPy 데이터를 모델 입력으로 바꿉니다.
- **주의점:** 한쪽의 in-place 변경이 다른 쪽에도 반영될 수 있습니다.
- **공식 문서:** [torch.from_numpy](https://docs.pytorch.org/docs/stable/generated/torch.from_numpy.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 목적함수·학습 update

- **원문 위치:** §2: winning ticket definition / §2, Figure 1: iterative pruning procedure
- **이 셀의 책임:** Task 1 / Task 1–2
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 초기값·mask 보존 / train→prune→rewind. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear,F.relu,Tensor.detach,torch.optim.Adam,Module.parameters,Optimizer.zero_grad,F.cross_entropy,Tensor.backward,Optimizer.step,Tensor.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 10개 · 수식 7개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>F.relu(input, inplace=False)</code></summary>

#### `F.relu(input, inplace=False)`

- **역할:** 음수 activation을 0으로 만드는 ReLU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 선형 층 사이에 비선형성을 추가합니다.
- **주의점:** 음수 구간 gradient는 0이며 in-place 연산은 autograd 값을 덮어쓸 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.relu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class TicketNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(32, 64)
        self.out = nn.Linear(64, 4)

    def forward(self, x):
        return self.out(F.relu(self.hidden(x)))


dense = TicketNet()
initial = {k: v.detach().clone() for k, v in dense.state_dict().items()}
optimizer = torch.optim.Adam(dense.parameters(), lr=0.025)
for _ in range(35):
    optimizer.zero_grad()
    loss = F.cross_entropy(dense(features[train_idx]), labels[train_idx])
    loss.backward()
    optimizer.step()
threshold = torch.quantile(dense.hidden.weight.detach().abs(), 0.70)
mask = (dense.hidden.weight.detach().abs() > threshold).float()
assert 0.28 < mask.mean().item() < 0.32

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §2: winning ticket definition / §2, Figure 1: iterative pruning procedure / §3: experimental methodology
- **이 셀의 책임:** Task 1 / Task 1–2 / Task 2
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 초기값·mask 보존 / train→prune→rewind / 동일 mask·step과 세 fresh optimizer. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.no_grad -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
ticket = TicketNet()
ticket.load_state_dict(initial)
with torch.no_grad():
    ticket.hidden.weight.mul_(mask)


def train_masked(model, mask, steps=45):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.025)
    history = []
    for _ in range(steps):
        optimizer.zero_grad()
        loss = F.cross_entropy(model(features[train_idx]), labels[train_idx])
        loss.backward()
        model.hidden.weight.grad.mul_(mask)
        optimizer.step()
        with torch.no_grad():
            model.hidden.weight.mul_(mask)
        history.append(float(loss.detach()))
    return history


ticket_history = train_masked(ticket, mask)
assert (ticket.hidden.weight.detach()[mask == 0] == 0).all() and ticket_history[
    -1
] < ticket_history[0]

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §4: winning tickets results
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** rewind ticket/random sparse/dense 비교. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
with torch.no_grad():
    ticket_accuracy = (
        (ticket(features[test_idx]).argmax(-1) == labels[test_idx])
        .float()
        .mean()
        .item()
    )
dense_count = sum(p.numel() for p in ticket.parameters())
active_count = (
    int(mask.sum())
    + ticket.hidden.bias.numel()
    + ticket.out.weight.numel()
    + ticket.out.bias.numel()
)
active_ratio = active_count / dense_count
assert ticket_accuracy > 0.75 and active_ratio < 0.45
plt.plot(ticket_history)
plt.title(
    f"rewound sparse ticket: acc={ticket_accuracy:.1%}, active={active_ratio:.1%}"
)
plt.show()
print(f"active parameters={active_ratio:.1%}, accuracy={ticket_accuracy:.1%}")

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2: winning ticket definition / §2, Figure 1: iterative pruning procedure / §3: experimental methodology / §4: winning tickets results
- **이 셀의 책임:** Task 1 / Task 1–2 / Task 2 / Task 3
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 초기값·mask 보존 / train→prune→rewind / 동일 mask·step과 세 fresh optimizer / rewind ticket/random sparse/dense 비교. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,torch.cat,torch.equal,Tensor.flatten -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

<details>
<summary><code>tensor.flatten(start_dim=0, end_dim=-1)</code></summary>

#### `tensor.flatten(start_dim=0, end_dim=-1)`

- **역할:** 지정한 연속 축들을 하나의 축으로 합칩니다.
- **입력·반환:** 축 범위를 받고 평탄화된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** feature map을 classifier 입력이나 토큰 열로 바꿉니다.
- **주의점:** batch 축까지 실수로 합치지 않도록 `start_dim`을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.flatten](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.flatten.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    keep_ratio: float = 0.3
    learning_rate: float = 0.025
    mask_finding_steps: int = 8
    comparison_steps: int = 10


def prepare_batch() -> tuple[Tensor, Tensor]:
    return features[train_idx], labels[train_idx]


class LotteryTicketPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.linear = nn.Linear(features.shape[1], 48)
        self.classifier = nn.Linear(48, teacher_logits.shape[1])
        self._initial_state = {
            name: parameter.detach().clone()
            for name, parameter in self.named_parameters()
        }
        self._masks = {
            name: torch.ones_like(parameter)
            for name, parameter in self.named_parameters()
        }
        self.rewind_verified = False
        self.optimizer_reset_count = 0

    def forward(self, inputs: Tensor) -> Tensor:
        return self.classifier(F.relu(self.linear(inputs)))

    def compute_loss(self, logits: Tensor, targets: Tensor) -> Tensor:
        return F.cross_entropy(logits, targets)

    def training_step(self, inputs: Tensor, targets: Tensor) -> Tensor:
        return self.compute_loss(self(inputs), targets)

    def parameter_state(self) -> dict[str, Tensor]:
        return {
            name: parameter.detach().clone()
            for name, parameter in self.named_parameters()
        }

    def load_parameter_state(self, state: dict[str, Tensor]) -> None:
        with torch.no_grad():
            for name, parameter in self.named_parameters():
                parameter.copy_(state[name].to(parameter.device))

    def mask_state(self) -> dict[str, Tensor]:
        return {name: mask.detach().clone() for name, mask in self._masks.items()}

    def install_masks(self, masks: dict[str, Tensor]) -> None:
        self._masks = {
            name: masks[name].to(parameter.device).clone()
            for name, parameter in self.named_parameters()
        }
        self.enforce_mask()

    def prune_and_rewind(self) -> None:
        with torch.no_grad():
            prunable = [
                parameter.abs().flatten()
                for parameter in self.parameters()
                if parameter.ndim >= 2
            ]
            threshold = torch.quantile(
                torch.cat(prunable), 1.0 - self.config.keep_ratio
            )
            for name, parameter in self.named_parameters():
                if parameter.ndim >= 2:
                    self._masks[name] = (parameter.abs() >= threshold).float()
                initial = self._initial_state[name].to(parameter.device)
                mask = self._masks[name].to(parameter.device)
                parameter.copy_(initial * mask)
            self.rewind_verified = all(
                torch.equal(
                    parameter,
                    self._initial_state[name].to(parameter.device)
                    * self._masks[name].to(parameter.device),
                )
                for name, parameter in self.named_parameters()
            )

    def mask_gradients(self) -> None:
        for name, parameter in self.named_parameters():
            if parameter.grad is not None:
                parameter.grad.mul_(self._masks[name].to(parameter.device))

    def enforce_mask(self) -> None:
        with torch.no_grad():
            for name, parameter in self.named_parameters():
                parameter.mul_(self._masks[name].to(parameter.device))

    def sparsity(self) -> Tensor:
        weight_masks = [
            mask.flatten()
            for name, mask in self._masks.items()
            if name.endswith("weight")
        ]
        return 1.0 - torch.cat(weight_masks).mean()


@dataclass
class LotteryComparison:
    ticket: LotteryTicketPortfolioModel
    random_control: LotteryTicketPortfolioModel
    dense_baseline: LotteryTicketPortfolioModel
    mask_finding_history: list[float]
    histories: dict[str, list[float]]
    fresh_optimizer_count: int
    random_control_differs: bool

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2: winning ticket definition / §2, Figure 1: iterative pruning procedure / §3: experimental methodology
- **이 셀의 책임:** Task 1 / Task 1–2 / Task 2
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 초기값·mask 보존 / train→prune→rewind / 동일 mask·step과 세 fresh optimizer. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def train_comparison_model(
    model: LotteryTicketPortfolioModel,
    inputs: Tensor,
    targets: Tensor,
    steps: int,
) -> tuple[list[float], int]:
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=model.config.learning_rate,
    )
    fresh_optimizer = int(len(optimizer.state) == 0)
    history = []
    for _ in range(steps):
        loss = model.training_step(inputs, targets)
        optimizer.zero_grad()
        loss.backward()
        model.mask_gradients()
        optimizer.step()
        model.enforce_mask()
        history.append(float(loss.detach().cpu()))
    model.optimizer_reset_count = fresh_optimizer
    return history, fresh_optimizer


def fit_portfolio_model(
    model: LotteryTicketPortfolioModel,
) -> LotteryComparison:
    inputs, targets = prepare_batch()
    initial_state = model.parameter_state()
    dense_optimizer = torch.optim.Adam(
        model.parameters(),
        lr=model.config.learning_rate,
    )
    mask_finding_history = []
    for _ in range(model.config.mask_finding_steps):
        loss = model.training_step(inputs, targets)
        dense_optimizer.zero_grad()
        loss.backward()
        dense_optimizer.step()
        mask_finding_history.append(float(loss.detach().cpu()))
    model.prune_and_rewind()

    shared_masks = model.mask_state()
    dense_baseline = LotteryTicketPortfolioModel(model.config)
    dense_baseline.load_parameter_state(initial_state)
    random_control = LotteryTicketPortfolioModel(model.config)
    random_control.install_masks(shared_masks)
    random_control_differs = any(
        not torch.equal(
            ticket_parameter,
            control_parameter,
        )
        for ticket_parameter, control_parameter in zip(
            model.parameters(),
            random_control.parameters(),
        )
    )

    histories = {}
    fresh_optimizer_count = 0
    candidates = {
        "ticket": model,
        "random_control": random_control,
        "dense_baseline": dense_baseline,
    }
    for name, candidate in candidates.items():
        history, fresh = train_comparison_model(
            candidate,
            inputs,
            targets,
            model.config.comparison_steps,
        )
        histories[name] = history
        fresh_optimizer_count += fresh
    return LotteryComparison(
        ticket=model,
        random_control=random_control,
        dense_baseline=dense_baseline,
        mask_finding_history=mask_finding_history,
        histories=histories,
        fresh_optimizer_count=fresh_optimizer_count,
        random_control_differs=random_control_differs,
    )

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §4: winning tickets results
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** features [B,D] + mask [O,I] → logits [B,K]와 sparsity scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** rewind ticket/random sparse/dense 비교. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.argmax,Tensor.cpu,np.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

<details>
<summary><code>tensor.cpu()</code></summary>

#### `tensor.cpu()`

- **역할:** 텐서를 CPU 메모리로 옮깁니다.
- **입력·반환:** 텐서를 받아 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** NumPy 변환이나 CPU 기반 평가 도구에 값을 전달합니다.
- **주의점:** GPU 텐서 이동은 동기화·복사 비용을 만들 수 있습니다.
- **공식 문서:** [Tensor.cpu](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.cpu.html)

</details>

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def evaluate_portfolio_model(
    comparison: LotteryComparison,
) -> dict[str, float]:
    accuracies = {}
    with torch.no_grad():
        candidates = {
            "ticket": comparison.ticket,
            "random_control": comparison.random_control,
            "dense_baseline": comparison.dense_baseline,
        }
        for name, candidate in candidates.items():
            logits = candidate(features[test_idx])
            accuracy = (logits.argmax(1) == labels[test_idx]).float().mean()
            accuracies[name] = float(accuracy.cpu())
        ticket_sparsity = float(comparison.ticket.sparsity().cpu())
        control_sparsity = float(comparison.random_control.sparsity().cpu())
    ticket_loss = comparison.histories["ticket"][-1]
    control_loss = comparison.histories["random_control"][-1]
    dense_loss = comparison.histories["dense_baseline"][-1]
    winning_condition = (
        accuracies["ticket"] >= accuracies["random_control"]
        and accuracies["ticket"] >= accuracies["dense_baseline"] - 0.05
        and ticket_loss <= control_loss
    )
    return {
        "ticket_test_accuracy": accuracies["ticket"],
        "random_control_test_accuracy": accuracies["random_control"],
        "dense_baseline_test_accuracy": accuracies["dense_baseline"],
        "ticket_minus_random_accuracy": (
            accuracies["ticket"] - accuracies["random_control"]
        ),
        "ticket_minus_dense_accuracy": (
            accuracies["ticket"] - accuracies["dense_baseline"]
        ),
        "ticket_final_train_loss": ticket_loss,
        "random_control_final_train_loss": control_loss,
        "dense_baseline_final_train_loss": dense_loss,
        "random_minus_ticket_train_loss": control_loss - ticket_loss,
        "ticket_weight_sparsity": ticket_sparsity,
        "control_weight_sparsity": control_sparsity,
        "matched_comparison_steps": float(comparison.ticket.config.comparison_steps),
        "full_tensor_rewind_verified": float(comparison.ticket.rewind_verified),
        "fresh_comparison_optimizers": float(comparison.fresh_optimizer_count),
        "random_control_differs": float(comparison.random_control_differs),
        "winning_ticket_condition": float(winning_condition),
    }


portfolio_model = LotteryTicketPortfolioModel(PortfolioConfig())
portfolio_comparison = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_comparison)
all_histories = [
    portfolio_comparison.mask_finding_history,
    *portfolio_comparison.histories.values(),
]
assert all(np.isfinite(history).all() for history in all_histories)
assert portfolio_metrics["fresh_comparison_optimizers"] == 3.0
assert portfolio_metrics["random_control_differs"] == 1.0
assert (
    abs(
        portfolio_metrics["ticket_weight_sparsity"]
        - portfolio_metrics["control_weight_sparsity"]
    )
    < 1e-8
)
assert portfolio_metrics["winning_ticket_condition"] == 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 MNIST/CIFAR의 반복 magnitude pruning과 여러 seed를 비교했다. 실습은 한 번의 pruning과 작은 MLP다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.